# 🏸 badak: count your badminton strokes from a Garmin `.fit` file

Run the cells from top to bottom. Nothing is installed on your computer; everything runs in this Colab session.

You need a `.fit` file that contains **raw accelerometer + gyroscope data**, recorded with the watch on your racket wrist (right-handed). A normal Garmin activity file only stores 1-second summaries and will not work.

## 1. Two small installs
`fitparse` reads Garmin files. The models were saved with scikit-learn 1.9.1, so we install that exact version to load them safely. Everything else (numpy, scipy, PyTorch, matplotlib) is already in Colab.

In [ ]:
!pip install -q fitparse scikit-learn==1.9.1

## 2. Download the models and code from Hugging Face

In [ ]:
from huggingface_hub import snapshot_download
import sys

repo = snapshot_download("omayib/badak")
sys.path.insert(0, repo)

from badminsens.predict import analyze, print_summary
from badminsens.plot import plot_session
print("ready:", repo)

## 3. Upload your `.fit` file

In [ ]:
from google.colab import files

uploaded = files.upload()
fit = next(iter(uploaded))
print("using", fit)

## 4. Count the strokes
`model="default"` is the SVM + CNN + Transformer ensemble. Other options: `"svm"`, `"gru"`, `"rf"`, `"et"`, `"hgb"`, `"mlp"`, `"cnn"`, `"transformer"`, `"lstm"`, `"cnn_lstm"`, `"tcn"`, `"logreg"`.

In [ ]:
rows, summary = analyze(fit, model="default")
print_summary(summary, "default model")

## 5. Heart rate with every stroke on the timeline

In [ ]:
fig = plot_session(fit, rows, "default")

## 6. (Optional) Compare all models

In [ ]:
import pandas as pd

names = ["default", "svm", "gru", "rf", "et", "hgb", "mlp", "cnn", "transformer"]
table = {}
for n in names:
    _, s = analyze(fit, model=n)
    table[n] = {k: v["count"] for k, v in s["strokes"].items()} | {"Other": s["other"]["count"]}
pd.DataFrame(table).T

## 7. Download the per-swing results

In [ ]:
import pandas as pd

pd.DataFrame(rows).to_csv("strokes.csv", index=False)
files.download("strokes.csv")